# [데모 D] 교수법 도구 체험장 — 기존 도구 6가지를 한 번씩 써 보기

> 개선 계획의 1~6단계에서 쓰기로 한 **기존 도구**를 하나씩 직접 돌려 보는 노트북이다. 각 절은 **무엇인가 → 왜 쓰나 → 해 보기 → 볼 지점 → 사내망에서 확인할 것** 순서다.
> 이 노트북을 사내망 PC에서 그대로 실행하면 계획의 **0단계(사내망 점검)**가 된다. 맨 아래 점검표를 채운다.

| 절 | 계획 단계 | 도구 | 수업 화면에 나오나 |
|---|---|---|---|
| 1 | 제작 기반 | Jupytext, nbstripout | 아니오 — 강사 쪽 관리 도구 |
| 2 | 검증 루프 | ipytest | 예 — 실습의 "검증" 칸 |
| 3 | 그림 | Graphviz, diagrams, archify, Mermaid | 예 — 개념도·구성도 |
| 4 | 개념 시각화 | tiktoken, 청크 색칠, plotly | 예 — 1.2 토큰, 2.1 RAG |
| 5 | 퀴즈 | jupyterquiz | 예 — 개념 확인 |
| 6 | 도구화 | `lessons/assets` | 아니오 — 다른 강의로 옮길 묶음 |

**준비:** 프로젝트 루트에서 `uv sync --extra notebooks --extra teaching`. 4-3절만 임베딩 API를 호출한다. 나머지는 LLM을 부르지 않는다.

## 1. 제작 기반 — Jupytext, nbstripout

### 1-1. Jupytext: 노트북을 `.py`로 보기

- **무엇인가:** `.ipynb`(JSON)를 사람이 읽는 `.py` 텍스트로 바꾸고, 둘을 짝지어 동기화하는 도구.
- **왜 쓰나:** `.ipynb`는 JSON이라 CodeMate·Claude가 고치거나 `git diff`로 비교하기 어렵다. `.py`로 보면 교안이 코드처럼 읽힌다.

In [ ]:
import jupytext

nb = jupytext.read("demo_a_slide_flow.ipynb")
text = jupytext.writes(nb, fmt="py:percent")
body = text[text.index("# %%") :]  # 맨 위 메타데이터 머리말은 건너뛴다
print("\n".join(body.splitlines()[:40]))

**볼 지점:** 마크다운 셀은 `# %% [markdown]` 아래 주석으로, 코드 셀은 `# %%` 아래 코드로 바뀐다. 셀 경계가 그대로 남아 다시 노트북으로 되돌릴 수 있다.

실제로 짝을 지으려면(아직 하지 않음): `uv run jupytext --set-formats ipynb,py:percent lessons/demo_a_slide_flow.ipynb`. 그러면 저장할 때마다 `.py`가 함께 갱신된다.

### 1-2. nbstripout: 커밋 전에 출력 지우기

- **무엇인가:** 노트북에서 실행 결과(출력)와 실행 번호를 지우는 도구. Git에 필터로 걸면 커밋할 때 자동으로 지운다.
- **왜 쓰나:** 출력에는 키·개인정보·큰 이미지가 섞일 수 있다. "노트북 출력은 커밋하지 않는다" 규칙을 사람이 아니라 도구가 지키게 한다.

In [ ]:
import subprocess
import sys
import tempfile
from pathlib import Path

import nbformat

sample = nbformat.v4.new_notebook(
    cells=[
        nbformat.v4.new_code_cell(
            "print(api_key)",
            execution_count=7,
            outputs=[
                nbformat.v4.new_output(
                    "stream", name="stdout", text="sk-이런-값이-출력에-남아-있다\n"
                )
            ],
        )
    ]
)

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "sample.ipynb"
    nbformat.write(sample, path)
    before = nbformat.read(path, as_version=4).cells[0]
    subprocess.run([sys.executable, "-m", "nbstripout", str(path)], check=True)
    after = nbformat.read(path, as_version=4).cells[0]

print("지우기 전: 출력", len(before.outputs), "개, 실행 번호", before.execution_count)
print("지운 뒤  : 출력", len(after.outputs), "개, 실행 번호", after.execution_count)

**볼 지점:** 코드는 그대로 두고 출력과 실행 번호만 사라진다.

Git 필터로 거는 명령(설정 변경이라 아직 하지 않음): `uv run nbstripout --install --attributes .gitattributes`.

**사내망에서 확인할 것:** 없음. 강사 PC에서만 쓴다.

## 2. 검증 루프 — ipytest

- **무엇인가:** 노트북 셀 안에서 pytest를 돌리는 도구. 셀 맨 위에 `%%ipytest`를 붙이면 그 셀의 테스트가 실행된다.
- **왜 쓰나:** "AI가 만든 코드를 테스트로 확인하고 다시 지시한다"(교육 목표 4)를 눈대중이 아닌 **통과/실패**로 보여 준다. 데모 C의 점수표를 표준 도구로 바꾼 것이다.

아래 함수는 데모 B 원본의 언어 판별식을 그대로 옮긴 것이다. 테스트가 이 식의 약점을 잡아내는지 본다.

In [ ]:
import ipytest

ipytest.autoconfig()


def detect_lang_from_reply(reply: str) -> str:
    """데모 B 원본(detect_language)의 판별식 그대로."""
    return "ko" if "ko" in reply.lower() else "en"

In [ ]:
%%ipytest -q --tb=short
import pytest


@pytest.mark.parametrize(
    "reply, expected",
    [
        ("ko", "ko"),
        ("en", "en"),
        ("Korean", "ko"),
        ("English", "en"),
        ("The text is English, not Korean.", "en"),
    ],
)
def test_detect_lang(reply, expected):
    assert detect_lang_from_reply(reply) == expected

**볼 지점:** 5개 중 1개가 실패한다. 실패 메시지에 어떤 입력에서 무엇이 나왔는지가 찍힌다. 이 빨간 줄이 **다시 지시할 근거**다.

**CodeMate 지시문 예시**

> 위 테스트를 모두 통과하도록 `detect_lang_from_reply` 함수만 고쳐 줘. 테스트는 고치지 마.

고친 뒤 두 셀을 다시 실행해 모두 통과하는지 본다. 테스트가 실패해도 노트북 실행은 멈추지 않는다.

**사내망에서 확인할 것:** `%%ipytest` 셀이 VS Code에서 실행되고 결과가 출력되는지.

## 3. 그림 — 네 가지 방법

| 방법 | 누가 만드나 | 결과 | 강의장 PC에 필요한 것 |
|---|---|---|---|
| Graphviz `.dot` | 강사가 텍스트로 작성 | SVG | 없음 (미리 만든 SVG를 넣는 경우) |
| diagrams | 강사가 Python으로 작성 | PNG | 없음 (출력이 노트북에 저장됨) |
| archify | AI 에이전트가 JSON 작성 → 렌더러 | 움직이는 HTML | 없음 (한 파일에 다 들어 있음) |
| Mermaid | 마크다운 셀에 텍스트 | VS Code가 그림 | VS Code 1.121 이상 |

### 3-1. Graphviz

- **무엇인가:** 노드와 화살표를 텍스트로 적으면 배치를 자동으로 해 주는 오래된 표준 도구.
- **왜 쓰나:** 흐름도가 텍스트라 고치기 쉽고 결과가 매번 같다. 저장소에 이미 틀(`docs/templates/flow_diagram.dot`)이 있다.

강사 PC에서 `dot -Tsvg docs/templates/flow_diagram.dot -o lessons/assets/diagrams/flow_diagram.svg`로 미리 만든 그림을 보여 준다.

In [ ]:
from IPython.display import SVG

SVG(filename="../assets/diagrams/flow_diagram.svg")

같은 그림을 노트북에서 바로 그릴 수도 있다. 단, 이 PC에 Graphviz 프로그램(`dot`)이 있어야 한다. 강의장 PC에는 없을 가능성이 크므로 위처럼 미리 만든 SVG를 쓰는 것이 기본이다.

In [ ]:
import graphviz

try:
    diagram = graphviz.Source(
        Path("../../docs/templates/flow_diagram.dot").read_text(encoding="utf-8")
    )
    diagram.pipe(format="svg")  # dot 프로그램이 없으면 여기서 오류가 난다
    display(diagram)
except graphviz.ExecutableNotFound:
    print("이 PC에는 Graphviz 프로그램(dot)이 없다 → 미리 만든 SVG를 쓴다.")

### 3-2. diagrams (mingrammer)

- **무엇인가:** Python 코드로 시스템 구성도를 그리는 라이브러리. 서버·DB·사용자 아이콘이 들어 있다.
- **왜 쓰나:** 사내 게이트웨이·RAGaaS 같은 구성도를 코드로 관리한다. 내부적으로 Graphviz를 쓰므로 **강사 PC에서** 그린다.

In [ ]:
from IPython.display import Image
from diagrams import Cluster, Diagram, Edge
from diagrams.generic.database import SQL
from diagrams.onprem.client import User
from diagrams.programming.language import Python

with tempfile.TemporaryDirectory() as tmp:
    out = Path(tmp) / "rag_layout"
    with Diagram(
        "사내 RAG 실습 구성",
        filename=str(out),
        outformat="png",
        show=False,
        direction="LR",
    ):
        user = User("수강생")
        with Cluster("강의장 PC"):
            app = Python("노트북 / 앱")
        with Cluster("사내망"):
            llm = Python("LLM 게이트웨이")
            rag = SQL("RAGaaS")
        user >> app >> Edge(label="질문") >> llm
        app >> Edge(label="검색") >> rag
    png = out.with_suffix(".png").read_bytes()

Image(data=png, width=700)

**볼 지점:** 아이콘은 모양을 보여 주는 예시일 뿐이다. 사내 LLM 게이트웨이나 RAGaaS 전용 아이콘은 없어서 Python·DB 아이콘을 빌려 썼다.

### 3-3. archify

- **무엇인가:** AI 에이전트(Claude Code, Codex 등)에 설치하는 스킬. 에이전트가 JSON으로 그림 내용을 적으면 렌더러가 **움직이는 흐름을 담은 HTML 한 파일**로 만든다. 레이아웃 검사도 해 준다.
- **왜 쓰나:** 흐름·시퀀스·상태 전이 그림을 말로 시켜서 만든다. 결과물에 외부 연결이 없어 사내망에서도 열린다.

아래는 데모 A의 Tool Calling 수동 루프를 JSON(`assets/diagrams/tool-calling.workflow.json`)으로 적고 렌더러로 만든 결과다. 한 페이지짜리 HTML이라 iframe으로 격리해서 넣는다.

In [ ]:
import html as html_lib

from IPython.display import HTML

page = Path("../assets/diagrams/tool-calling.html").read_text(encoding="utf-8")
HTML(
    f'<div><iframe srcdoc="{html_lib.escape(page)}" style="width:100%;height:720px;border:1px solid #d0d7de;border-radius:8px"></iframe></div>'
)

**볼 지점:** 그림 안의 버튼으로 흐름 애니메이션·테마·내보내기(PNG/SVG)를 쓸 수 있다. 파일 크기는 약 0.8MB(글꼴 포함)다.

다시 만드는 법(강사 PC, Node 18 이상): archify 저장소의 `archify/` 폴더에서
`node bin/archify.mjs deliver workflow <JSON 경로> <HTML 경로> --quality standard --json`.
처음 만든 JSON은 레이아웃 검사에서 선이 겹친다는 지적을 받아 고쳤다. 검사가 "조용히 틀린 그림"을 막아 준다.

### 3-4. Mermaid (VS Code 내장)

- **무엇인가:** 마크다운 안에 텍스트로 흐름도를 적는 문법. VS Code 1.121(2026-05)부터 노트북 마크다운 셀에서 바로 그림으로 보인다.
- **왜 쓰나:** 설치할 것이 없다. LangGraph의 `draw_mermaid()` 결과를 붙여 넣으면 그래프 구조도가 된다.

아래 마크다운 셀은 **VS Code에서만** 그림으로 보인다. 다른 곳에서는 글자로 보인다.

```mermaid
flowchart LR
    Q[질문] --> I[llm_with_tools.invoke]
    I --> D{tool_calls?}
    D -- 있음 --> E[도구 실행]
    E --> T[ToolMessage 추가]
    T --> I
    D -- 없음 --> A[최종 답변]
```

LangGraph 그래프에서 Mermaid 텍스트를 뽑는다. LLM 없이 도는 작은 그래프다. 출력 글자를 마크다운 셀의 ```` ```mermaid ```` 블록에 붙이면 그림이 된다. **`draw_mermaid_png()`는 외부 서버를 불러 사내망에서 실패하므로 쓰지 않는다.**

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import END, START, StateGraph


class S(TypedDict):
    text: str


builder = StateGraph(S)
builder.add_node("detect", lambda s: {"text": s["text"]})
builder.add_node("korean", lambda s: {"text": "ko"})
builder.add_node("english", lambda s: {"text": "en"})
builder.add_edge(START, "detect")
builder.add_conditional_edges("detect", lambda s: "korean", ["korean", "english"])
builder.add_edge("korean", END)
builder.add_edge("english", END)

print(builder.compile().get_graph().draw_mermaid())

**사내망에서 확인할 것:** SVG·PNG 그림이 보이는지, archify iframe이 움직이는지, Mermaid 마크다운 셀이 그림으로 바뀌는지(VS Code 버전 확인: 메뉴 Code/Help → About).

## 4. 개념 시각화 — 토큰, 청크, 임베딩

### 4-1. 토큰 색칠 (1.2 LLM 작동 원리)

- **무엇인가:** tiktoken은 OpenAI 계열 모델의 토크나이저다. 문장이 어떤 조각(토큰)으로 쪼개지는지 색으로 보여 준다.
- **왜 쓰나:** "한국어가 영어보다 토큰을 더 쓴다", "컨텍스트 윈도우는 글자 수가 아니라 토큰 수다"를 눈으로 보여 준다.
- **한계:** 사내 모델의 토크나이저는 다를 수 있다(미확인). 개념 설명용이다. tiktoken은 처음 쓸 때 토큰 사전을 인터넷에서 받으므로, 사내망에서는 미리 받아 둔 폴더를 `TIKTOKEN_CACHE_DIR`로 지정해야 한다.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("o200k_base")
COLORS = ["#fff8c5", "#ddf4ff", "#dafbe1", "#fbefff", "#ffebe9"]


def token_html(sentence: str) -> str:
    pieces = [enc.decode([t]) for t in enc.encode(sentence)]
    spans = "".join(
        f'<span style="background:{COLORS[i % len(COLORS)]};padding:2px 1px;border-radius:3px">{html_lib.escape(p)}</span>'
        for i, p in enumerate(pieces)
    )
    return f'<div style="margin:6px 0;font-size:18px;color:#1f2328">{spans} <b style="color:#57606a">— {len(pieces)} 토큰, {len(sentence)} 글자</b></div>'


HTML(
    "".join(
        token_html(s)
        for s in [
            "LangGraph is a library for building stateful agents.",
            "LangGraph는 상태를 가진 에이전트를 만드는 라이브러리다.",
            "사내 게이트웨이로 LLM을 호출한다.",
        ]
    )
)

**볼 지점:** 같은 뜻인데 한국어 문장의 토큰 수가 다르다. 색 경계가 단어 경계와 일치하지 않는다.

### 4-2. 청크 색칠 (2.1 RAG 청킹)

- **무엇인가:** 텍스트 분할기가 문서를 어디서 자르는지 색으로 보여 준다. 직접 만든 몇 줄짜리 HTML이다.
- **왜 쓰나:** `chunk_size`와 `chunk_overlap`을 바꿨을 때 결과가 어떻게 달라지는지 바로 비교한다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

doc = Path("../../src/raw/Chapter02/06-capstone/data/ai_trends.md").read_text(
    encoding="utf-8"
)[:700]


def chunk_html(chunk_size: int, chunk_overlap: int) -> str:
    chunks = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    ).split_text(doc)
    boxes = "".join(
        f'<div style="background:{COLORS[i % len(COLORS)]};border-radius:6px;padding:6px;margin:4px 0;white-space:pre-wrap">'
        f"<b>#{i + 1} ({len(c)}자)</b> {html_lib.escape(c)}</div>"
        for i, c in enumerate(chunks)
    )
    return (
        f'<div style="font-size:14px;color:#1f2328;max-width:900px"><b>chunk_size={chunk_size}, '
        f"chunk_overlap={chunk_overlap} → {len(chunks)}개</b>{boxes}</div>"
    )


HTML(chunk_html(200, 0) + chunk_html(200, 60))

**볼 지점:** overlap을 주면 앞 청크의 끝부분이 다음 청크 앞에 다시 나온다. 분할기는 가능하면 문단·줄바꿈에서 자른다.

### 4-3. 임베딩 공간 (2.1 RAG 임베딩·검색) — 임베딩 API 호출

- **무엇인가:** 문장을 임베딩한 벡터를 PCA로 2차원에 눌러 점으로 찍는다. plotly는 마우스를 올리면 문장이 보이는 그래프를 인터넷 없이 그린다.
- **왜 쓰나:** "뜻이 비슷한 문장은 가까이 모인다", "질문과 가까운 문장을 찾는 것이 검색이다"를 눈으로 보여 준다.
- **한계:** 수천 차원을 2차원으로 누른 그림이라 거리가 정확히 보존되지는 않는다.

In [ ]:
import numpy as np
import plotly.express as px

from config.embeddings import get_embeddings

sentences = {
    "날씨": [
        "오늘 서울은 맑고 따뜻하다.",
        "내일 부산에 비가 온다.",
        "주말에 기온이 크게 떨어진다.",
    ],
    "요리": [
        "김치찌개는 돼지고기와 김치로 끓인다.",
        "파스타 면은 소금물에 삶는다.",
        "달걀말이는 약불에서 만다.",
    ],
    "LangGraph": [
        "노드는 State를 받아 일부를 돌려준다.",
        "조건부 엣지는 다음 노드를 고른다.",
        "체크포인터가 실행 상태를 저장한다.",
    ],
}
query = "그래프에서 다음 단계는 어떻게 정해지나?"

texts = [s for group in sentences.values() for s in group] + [query]
labels = [name for name, group in sentences.items() for _ in group] + ["질문"]
vectors = np.array(get_embeddings().embed_documents(texts))

centered = vectors - vectors.mean(axis=0)
_, _, vt = np.linalg.svd(centered, full_matrices=False)
xy = centered @ vt[:2].T  # PCA 2차원

fig = px.scatter(
    x=xy[:, 0],
    y=xy[:, 1],
    color=labels,
    hover_name=texts,
    labels={"x": "PC1", "y": "PC2", "color": "주제"},
    width=760,
    height=480,
)
fig.update_traces(marker_size=14)
fig.show()

같은 그림을 정적 이미지(matplotlib)로도 그린다. plotly가 안 보이는 환경의 예비용이다. 한글 글꼴 문제를 피하려고 점에는 번호만 붙이고 문장은 아래에 출력한다.

In [ ]:
import matplotlib.pyplot as plt

fig2, ax = plt.subplots(figsize=(7, 4.5))
for name in list(sentences) + ["질문"]:
    idx = [i for i, label in enumerate(labels) if label == name]
    ax.scatter(xy[idx, 0], xy[idx, 1], s=90, marker="*" if name == "질문" else "o")
for i, (x, y) in enumerate(xy):
    ax.annotate(str(i), (x, y), textcoords="offset points", xytext=(6, 4))
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
plt.show()

for i, (label, text) in enumerate(zip(labels, texts)):
    print(f"{i:2} [{label}] {text}")

**볼 지점:** 같은 주제끼리 모이고, 질문(별표)은 LangGraph 문장 쪽에 가깝다.

**사내망에서 확인할 것:** 토큰 사전 캐시 위치, 사내 임베딩(`EMBED_PROVIDER=samsung`)으로 같은 그림이 나오는지, plotly 그래프가 VS Code에서 보이는지(파일을 닫았다 다시 열었을 때도).

## 5. 퀴즈 — jupyterquiz

- **무엇인가:** 질문·보기·해설을 dict(또는 JSON 파일)로 적으면 클릭형 퀴즈로 보여 주는 도구. 퀴즈용 JS가 패키지 안에 들어 있다.
- **왜 쓰나:** 개념 확인을 수강생이 직접 눌러 보게 한다. 문제를 JSON 파일로 따로 관리할 수 있다.
- **한계:** 답안이 저장·수집되지 않는다. 출력이 `application/javascript` 형식인데 VS Code가 이를 그리는지는 미확인이다. 페이지에 수식 엔진이 이미 있는 특정 조건에서만 수식 스크립트를 CDN에서 불러오려 한다(퀴즈 JS에서 확인). 사내망에서는 그 요청만 실패하므로 수식(LaTeX)을 쓰지 않는다.

In [ ]:
from jupyterquiz import display_quiz

display_quiz(
    [
        {
            "question": "Tool Calling에서 get_weather 함수를 실제로 실행하는 것은 누구인가?",
            "type": "multiple_choice",
            "answers": [
                {
                    "answer": "LLM",
                    "correct": False,
                    "feedback": "모델은 도구 이름과 인자(tool_calls)만 만든다.",
                },
                {
                    "answer": "우리 코드(앱)",
                    "correct": True,
                    "feedback": "맞다. tool_map으로 함수를 찾아 앱이 실행한다.",
                },
                {
                    "answer": "LangChain 서버",
                    "correct": False,
                    "feedback": "외부 서버가 실행하지 않는다.",
                },
            ],
        },
        {
            "question": "State의 messages에 add_messages Reducer를 붙이면 노드가 돌려준 메시지는 어떻게 되나?",
            "type": "multiple_choice",
            "answers": [
                {
                    "answer": "기존 목록을 덮어쓴다",
                    "correct": False,
                    "feedback": "Reducer가 없는 language 필드가 덮어쓴다.",
                },
                {
                    "answer": "기존 목록 뒤에 누적된다",
                    "correct": True,
                    "feedback": "맞다. 그래서 대화 맥락이 쌓인다.",
                },
            ],
        },
    ]
)

**볼 지점:** 보기를 누르면 맞았는지와 해설이 바로 나온다. 객관식 말고도 복수 선택·숫자·글자 입력, 문제 은행, 플래시카드 등 11가지 형식은 `demo_e_quiz_types.ipynb`에 있다.

**사내망에서 확인할 것:** 퀴즈가 VS Code에서 보이고 눌리는지(문서상 미확인 항목).

## 6. 도구화 — 다른 강의에서도 쓰는 묶음

- **무엇인가:** 지금까지 만든 것(위젯 모듈, 슬라이드 묶음 생성기, 그림 틀, 규칙 문서)을 이 과정에서 떼어 내 다른 강의에도 쓰는 묶음으로 만드는 것.
- **왜 쓰나:** 강의가 바뀌어도 "슬라이드 인라인 + 실행 따라가기 + 검증 루프" 방식을 그대로 가져가기 위해서다.

지금 `lessons/assets`에 들어 있는 것:

In [ ]:
import sys

sys.path.insert(0, "..")  # lessons/ 아래의 assets 도구를 불러오기 위한 경로
import inspect

import assets.widgets as widgets

for name, fn in inspect.getmembers(widgets, inspect.isfunction):
    if not name.startswith("_") and fn.__module__ == widgets.__name__:
        print(f"{name:16} {inspect.getdoc(fn).splitlines()[0]}")

print()
for path in sorted(Path("../assets").rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts:
        print(f"{path.stat().st_size / 1e6:6.2f} MB  {path}")

**볼 지점:** 과정에 딸린 것은 `slides/`(PDF 변환 결과)와 `diagrams/`의 내용뿐이다. 나머지(`widgets.py`, `build_slide_deck.py`, 뷰어 틀)는 어느 강의에나 쓸 수 있다.

떼어 낼 때의 모양(아직 하지 않음): 위젯과 생성기를 작은 Python 패키지 하나로 만들고, `lessons/AGENTS.md`의 규칙을 AI 에이전트 스킬 하나로 만든다. 그러면 새 강의에서도 "이 PDF로 교안 노트북을 만들어 줘"라고 시킬 수 있다.

## 7. 사내망 점검표 (계획 0단계)

사내망 PC의 VS Code에서 이 노트북을 위에서부터 실행하며 체크한다. 안 되는 항목은 그 절의 예비 방법을 쓴다.

In [ ]:
from assets.widgets import show_checklist

show_checklist(
    [
        "VS Code 버전이 1.121 이상이다 (Mermaid 마크다운 셀)",
        "2절: %%ipytest 셀이 실행되고 실패 1개가 보인다",
        "3-1절: 미리 만든 SVG가 보인다",
        "3-2절: diagrams PNG가 보인다 (강의장 PC에 Graphviz가 없으면 오류 대신 안내가 나온다)",
        "3-3절: archify 그림이 iframe 안에서 움직인다",
        "3-4절: Mermaid 마크다운 셀이 그림으로 바뀐다",
        "4-1절: tiktoken이 인터넷 없이 동작한다 (TIKTOKEN_CACHE_DIR 준비)",
        "4-3절: 사내 임베딩으로 그림이 나온다",
        "4-3절: plotly 그래프가 보인다 (파일을 닫았다 다시 열어도)",
        "5절: 퀴즈가 보이고 눌린다",
        "데모 A~C의 위젯(슬라이드, 메시지 흐름, 그래프 실행, 코드 지도, 점수표)이 보이고 눌린다",
    ],
    title="사내망 점검표",
)